<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO_ChatTemplate_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install datasets evaluate transformers[sentencepiece]

In [18]:
!pip install --upgrade torchao

In [19]:
!pip install trl[GRPOTrainer]

In [20]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import re

In [21]:
# Dataset Prep
t_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:20]")
e_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:20]")

# train_dataset = t_dataset.rename_column("messages", "prompt")
# eval_dataset = e_dataset.rename_column("messages", "prompt")

In [22]:
# def generate_r1_prompt(tokenizer):
#     messages = [
#         {"role": "user", "content": "Hello, how are you?"},
#         {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
#         {"role": "user", "content": "I'd like to show off how chat templating works!"},
#     ]
#     return {
#         "prompt": tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)
#     }


# chat = [
#   {"role": "user", "content": "Hello, how are you?"},
#   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
#   {"role": "user", "content": "I'd like to show off how chat templating works!"},
# ]

def extract_prompt(example):
    messages = example["messages"]
    user_only = [m for m in messages if m["role"] == "user"]
    return {"prompt": [user_only[-1]]}

train_dataset = t_dataset.map(extract_prompt, remove_columns=t_dataset.column_names)
eval_dataset = e_dataset.map(extract_prompt, remove_columns=e_dataset.column_names)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M", device_map="auto")

tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

# tokenizer.apply_chat_template(chat, tokenize=False)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [23]:
# # Not really sure what the point of this is... since i already have a funciton
# train_dataset = train_dataset.map(lambda x: generate_r1_prompt(x))

In [24]:
def format_reward_func(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0.0
        # completion is a list of dicts with 'role' and 'content'
        text = completion[0]["content"].strip()
        if text and text[-1] in ".!?":
            score += 1.0
        if 20 <= len(text) <= 200:
            score += 0.5
        scores.append(score)
    return scores

In [25]:
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    num_generations=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    max_completion_length=128,
)

In [26]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    reward_funcs=format_reward_func,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


In [27]:
# print(train_dataset[0]["prompt"])

In [28]:
print(type(train_dataset[0]["prompt"]))
print(train_dataset[0]["prompt"])
print(tokenizer.chat_template)

<class 'list'>
[{'content': "I'm currently studying for the Law School Admission Test (LSAT) and I was hoping you could help me to identify the flaw in a practice argument I came across. The argument is: The company's profits have been declining for the past two years, but an analyst believes the company will return to profitability soon. She points to the fact that the company is the leading seller in its market, and that this is typically a very profitable position to be in. The analyst concludes that the company's dominant market share will cause its profits to increase. \n\nCan you identify what's wrong with this argument, specifically what logical fallacy it's committing?", 'role': 'user'}]
{% for message in messages %}{{ '<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>
' }}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [29]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate()